<a href="https://colab.research.google.com/github/viduladp/nova-ai-platform/blob/main/task3_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb==0.4.22 \
            sentence-transformers==2.7.0 \
            rank-bm25==0.2.2 \
            groq==0.13.0 \
            httpx==0.27.2 \
            ragas==0.1.7 \
            langchain==0.1.20 \
            langchain-community==0.0.38 \
            datasets==2.19.0 \
            pandas==2.1.4 \
            numpy==1.26.4 -q

print("✅ All packages installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.0/509.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 103.5 MB/s eta 

In [2]:
import os
import json
import time
import datetime
import numpy as np
import pandas as pd
from groq import Groq
from google.colab import userdata

# ChromaDB
import chromadb
from chromadb.utils import embedding_functions

# Embeddings + Re-ranker
from sentence_transformers import SentenceTransformer, CrossEncoder

# BM25
from rank_bm25 import BM25Okapi

# RAGAS
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)

# Groq client
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# Create folders
os.makedirs("chroma_db", exist_ok=True)
os.makedirs("data", exist_ok=True)

print("✅ All imports successful")

✅ All imports successful


In [5]:
from google.colab import files

print("Upload nova_mock_db.json from Task 2...")
uploaded = files.upload()  # select nova_mock_db.json

with open("nova_mock_db.json", "r") as f:
    db = json.load(f)

products = db["products"]
faqs     = db["faqs"]

print(f"✅ Database loaded")
print(f"   Products : {len(products)}")
print(f"   FAQs     : {len(faqs)}")

Upload nova_mock_db.json from Task 2...


Saving nova_mock_db.json to nova_mock_db (2).json
✅ Database loaded
   Products : 60
   FAQs     : 12


In [6]:
# Convert each product into a rich text chunk for indexing
# Each chunk contains ALL product info in a readable format
# This is what gets embedded and searched

def product_to_document(product: dict) -> dict:
    """
    Converts a product dict into a rich text document for indexing.
    Returns both the text chunk and metadata.
    """
    lines = []
    lines.append(f"Product: {product['name']}")
    lines.append(f"Product ID: {product['product_id']}")
    lines.append(f"Category: {product['category']}")
    lines.append(f"Price: ${product['price']}")
    lines.append(f"Rating: {product['rating']}/5")
    lines.append(f"Description: {product['description']}")

    # Category-specific fields
    if "ingredients" in product:
        lines.append(f"Ingredients: {', '.join(product['ingredients'])}")

    if "suitable_for" in product:
        lines.append(f"Suitable For: {', '.join(product['suitable_for'])}")

    if "available_sizes" in product:
        lines.append(f"Available Sizes: {', '.join(product['available_sizes'])}")

    if "material" in product:
        lines.append(f"Material: {product['material']}")

    if "finish" in product:
        lines.append(f"Finish: {product['finish']}")

    if "shades" in product:
        lines.append(f"Number of Shades: {len(product['shades'])}")

    if "size_ml" in product:
        lines.append(f"Size: {product['size_ml']}ml")

    if "heel_height" in product:
        lines.append(f"Heel Height: {product['heel_height']}")

    if "care" in product:
        lines.append(f"Care Instructions: {product['care']}")

    in_stock = product.get("stock", 0) > 0
    lines.append(f"In Stock: {'Yes' if in_stock else 'No'}")

    text = "\n".join(lines)

    return {
        "doc_id"    : product["product_id"],
        "text"      : text,
        "metadata"  : {
            "product_id" : product["product_id"],
            "name"       : product["name"],
            "category"   : product["category"],
            "price"      : str(product["price"]),
            "rating"     : str(product["rating"]),
            "in_stock"   : str(in_stock)
        }
    }

# Also add FAQs as documents
def faq_to_document(faq: dict) -> dict:
    text = f"FAQ: {faq['question']}\nAnswer: {faq['answer']}\nCategory: {faq['category']}"
    return {
        "doc_id"   : faq["id"],
        "text"     : text,
        "metadata" : {
            "product_id" : faq["id"],
            "name"       : faq["question"],
            "category"   : f"faq_{faq['category']}",
            "price"      : "0",
            "rating"     : "0",
            "in_stock"   : "N/A"
        }
    }

# Build all documents
product_docs = [product_to_document(p) for p in products]
faq_docs     = [faq_to_document(f) for f in faqs]
all_docs     = product_docs + faq_docs

print(f"✅ Documents prepared")
print(f"   Product chunks : {len(product_docs)}")
print(f"   FAQ chunks     : {len(faq_docs)}")
print(f"   Total chunks   : {len(all_docs)}")
print(f"\n📄 Sample document:")
print("-" * 50)
print(all_docs[0]["text"])

✅ Documents prepared
   Product chunks : 60
   FAQ chunks     : 12
   Total chunks   : 72

📄 Sample document:
--------------------------------------------------
Product: NOVA Vitamin C Serum
Product ID: SKU-1001
Category: skincare
Price: $100.59
Rating: 4.6/5
Description: Premium vitamin c serum by NOVA. Agent every development say.
Ingredients: Vitamin C, Retinol, Tea Tree Oil, Hyaluronic Acid
Suitable For: normal, oily, combination, dry
Size: 30ml
In Stock: Yes


In [7]:
# Load embedding model
print("⏳ Loading embedding model (BAAI/bge-small-en-v1.5)...")
print("   This may take 1-2 minutes on first run...")

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("✅ Embedding model loaded")
print(f"   Embedding dimensions: {embedding_model.get_sentence_embedding_dimension()}")

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if exists (clean rebuild)
try:
    chroma_client.delete_collection("nova_products")
    print("   Cleared existing collection")
except:
    pass

# Create fresh collection
collection = chroma_client.create_collection(
    name="nova_products",
    metadata={"hnsw:space": "cosine"}   # Cosine similarity for text
)

# Embed and index all documents in batches
print(f"\n⏳ Embedding {len(all_docs)} documents...")
BATCH_SIZE = 50

for i in range(0, len(all_docs), BATCH_SIZE):
    batch     = all_docs[i:i + BATCH_SIZE]
    texts     = [d["text"] for d in batch]
    ids       = [d["doc_id"] for d in batch]
    metadatas = [d["metadata"] for d in batch]

    # Generate embeddings
    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,   # Required for cosine similarity
        show_progress_bar=False
    ).tolist()

    # Add to ChromaDB
    collection.add(
        documents  = texts,
        embeddings = embeddings,
        ids        = ids,
        metadatas  = metadatas
    )

    print(f"   Indexed batch {i//BATCH_SIZE + 1} — "
          f"{min(i+BATCH_SIZE, len(all_docs))}/{len(all_docs)} docs")

print(f"\n✅ ChromaDB dense index built")
print(f"   Collection : nova_products")
print(f"   Documents  : {collection.count()}")
print(f"   Location   : ./chroma_db/")

⏳ Loading embedding model (BAAI/bge-small-en-v1.5)...
   This may take 1-2 minutes on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

✅ Embedding model loaded
   Embedding dimensions: 384


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



⏳ Embedding 72 documents...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


   Indexed batch 1 — 50/72 docs
   Indexed batch 2 — 72/72 docs

✅ ChromaDB dense index built
   Collection : nova_products
   Documents  : 72
   Location   : ./chroma_db/


In [8]:
# BM25 needs tokenized text (list of words per document)
print("⏳ Building BM25 sparse index...")

# Tokenize all documents
def tokenize(text: str) -> list:
    """Simple whitespace + lowercase tokenizer"""
    return text.lower().split()

all_texts     = [d["text"] for d in all_docs]
tokenized     = [tokenize(text) for text in all_texts]

# Build BM25 index
bm25_index = BM25Okapi(tokenized)

print(f"✅ BM25 sparse index built")
print(f"   Documents indexed : {len(tokenized)}")
print(f"   Avg doc length    : {np.mean([len(t) for t in tokenized]):.0f} tokens")

# Quick test
test_query  = "vitamin c serum niacinamide"
test_tokens = tokenize(test_query)
test_scores = bm25_index.get_scores(test_tokens)
top_idx     = np.argsort(test_scores)[::-1][:3]

print(f"\n🧪 BM25 test — query: '{test_query}'")
for idx in top_idx:
    print(f"   Score {test_scores[idx]:.2f} → {all_docs[idx]['doc_id']}: "
          f"{all_docs[idx]['text'][:60]}...")

⏳ Building BM25 sparse index...
✅ BM25 sparse index built
   Documents indexed : 72
   Avg doc length    : 36 tokens

🧪 BM25 test — query: 'vitamin c serum niacinamide'
   Score 10.52 → SKU-1001: Product: NOVA Vitamin C Serum
Product ID: SKU-1001
Category:...
   Score 8.91 → SKU-1005: Product: NOVA Niacinamide Toner
Product ID: SKU-1005
Categor...
   Score 5.60 → SKU-1006: Product: NOVA Hyaluronic Acid Serum
Product ID: SKU-1006
Cat...


In [9]:
print("⏳ Loading cross-encoder re-ranker...")
print("   Model: cross-encoder/ms-marco-MiniLM-L-6-v2")
print("   This may take 1-2 minutes...")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ Re-ranker loaded")

# Quick test
test_pairs = [
    ["vitamin c serum for oily skin", all_docs[0]["text"]],
    ["vitamin c serum for oily skin", all_docs[5]["text"]],
]
test_scores = reranker.predict(test_pairs)
print(f"\n🧪 Re-ranker test scores: {[round(s, 3) for s in test_scores]}")
print("   (Higher = more relevant)")

⏳ Loading cross-encoder re-ranker...
   Model: cross-encoder/ms-marco-MiniLM-L-6-v2
   This may take 1-2 minutes...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Re-ranker loaded

🧪 Re-ranker test scores: [5.932, 2.663]
   (Higher = more relevant)


In [10]:
def hybrid_retrieve(query: str, top_k: int = 3) -> list:
    """
    Hybrid retrieval: Dense (ChromaDB) + Sparse (BM25) + Re-ranking.

    Returns top_k most relevant document chunks.
    """

    # ── STEP 1: Dense Retrieval ────────────────────────────────
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    dense_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10,
        include=["documents", "metadatas", "distances"]
    )

    dense_docs = []
    for doc, meta, dist in zip(
        dense_results["documents"][0],
        dense_results["metadatas"][0],
        dense_results["distances"][0]
    ):
        dense_docs.append({
            "text"     : doc,
            "metadata" : meta,
            "score"    : 1 - dist,    # Convert distance to similarity
            "source"   : "dense"
        })

    # ── STEP 2: Sparse Retrieval (BM25) ───────────────────────
    query_tokens = tokenize(query)
    bm25_scores  = bm25_index.get_scores(query_tokens)
    top_bm25_idx = np.argsort(bm25_scores)[::-1][:10]

    sparse_docs = []
    max_bm25    = max(bm25_scores) if max(bm25_scores) > 0 else 1

    for idx in top_bm25_idx:
        if bm25_scores[idx] > 0:
            sparse_docs.append({
                "text"     : all_docs[idx]["text"],
                "metadata" : all_docs[idx]["metadata"],
                "score"    : bm25_scores[idx] / max_bm25,  # Normalize 0-1
                "source"   : "sparse"
            })

    # ── STEP 3: Merge + Deduplicate ────────────────────────────
    seen     = set()
    merged   = []

    for doc in dense_docs + sparse_docs:
        doc_id = doc["metadata"]["product_id"]
        if doc_id not in seen:
            seen.add(doc_id)
            merged.append(doc)

    # ── STEP 4: Cross-Encoder Re-ranking ──────────────────────
    if len(merged) == 0:
        return []

    rerank_pairs  = [[query, doc["text"]] for doc in merged]
    rerank_scores = reranker.predict(rerank_pairs)

    for doc, score in zip(merged, rerank_scores):
        doc["rerank_score"] = float(score)

    # Sort by re-rank score, take top_k
    reranked = sorted(merged, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]


print("✅ Hybrid retrieval function ready")
print("   Pipeline: Dense → Sparse → Merge → Re-rank → Top K")

✅ Hybrid retrieval function ready
   Pipeline: Dense → Sparse → Merge → Re-rank → Top K


In [11]:
def retrieve_and_answer(query: str, top_k: int = 3) -> dict:
    """
    Main RAG function — retrieves relevant product info and generates answer.
    This function is imported by Task 5.

    Returns:
        answer   : str   — LLM-generated answer
        sources  : list  — retrieved chunks used
        context  : str   — full context passed to LLM
    """
    start = time.time()

    # Step 1: Retrieve
    retrieved = hybrid_retrieve(query, top_k=top_k)

    if not retrieved:
        return {
            "answer"  : "I couldn't find specific product information for your query.",
            "sources" : [],
            "context" : "",
            "query"   : query,
            "latency_ms": round((time.time() - start) * 1000, 2)
        }

    # Step 2: Build context from retrieved chunks
    context_parts = []
    for i, doc in enumerate(retrieved, 1):
        context_parts.append(
            f"[Source {i} — {doc['metadata']['name']}]\n{doc['text']}"
        )
    context = "\n\n".join(context_parts)

    # Step 3: Generate answer with Groq
    system_prompt = """You are NOVA's Product Knowledge Assistant.
Answer the customer's question using ONLY the product information provided.
Be specific, accurate, and helpful. Mention the product name in your answer.
If the information isn't in the context, say so honestly.
Keep answers concise — 2-4 sentences maximum."""

    user_prompt = f"""Customer Question: {query}

Product Information:
{context}

Answer the customer's question based on the above product information."""

    response = client.chat.completions.create(
        model      = "llama-3.3-70b-versatile",
        messages   = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens  = 300,
        temperature = 0.2    # Low temp = factual, consistent
    )

    answer = response.choices[0].message.content

    return {
        "query"     : query,
        "answer"    : answer,
        "sources"   : [
            {
                "product_id"   : d["metadata"]["product_id"],
                "name"         : d["metadata"]["name"],
                "category"     : d["metadata"]["category"],
                "rerank_score" : round(d["rerank_score"], 3)
            }
            for d in retrieved
        ],
        "context"    : context,
        "latency_ms" : round((time.time() - start) * 1000, 2)
    }


print("✅ RAG answer function ready")
print("   Import in Task 5 with: from rag_module import retrieve_and_answer")

✅ RAG answer function ready
   Import in Task 5 with: from rag_module import retrieve_and_answer


In [12]:
test_queries = [
    "Does your vitamin C serum contain niacinamide?",
    "What sizes are available for the wrap dress?",
    "Which products are suitable for sensitive skin?",
    "What is your return policy?",
    "Do you have any cruelty-free makeup products?"
]

print("🧪 TESTING RAG PIPELINE — 5 Product Queries")
print("=" * 65)

rag_test_results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n📨 Query {i}: {query}")
    print("-" * 65)

    result = retrieve_and_answer(query)

    print(f"💬 Answer:\n   {result['answer']}")
    print(f"\n📚 Sources retrieved:")
    for src in result["sources"]:
        print(f"   • {src['name']} ({src['product_id']}) "
              f"— relevance: {src['rerank_score']:.3f}")
    print(f"⚡ Latency: {result['latency_ms']} ms")

    rag_test_results.append(result)

print("\n✅ All 5 queries answered successfully")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🧪 TESTING RAG PIPELINE — 5 Product Queries

📨 Query 1: Does your vitamin C serum contain niacinamide?
-----------------------------------------------------------------
💬 Answer:
   The NOVA Vitamin C Serum does not contain niacinamide. According to the product information, the ingredients of the NOVA Vitamin C Serum are Vitamin C, Retinol, Tea Tree Oil, and Hyaluronic Acid. If you're looking for a product with niacinamide, you might consider the NOVA Niacinamide Toner instead.

📚 Sources retrieved:
   • NOVA Vitamin C Serum (SKU-1001) — relevance: -3.663
   • NOVA Niacinamide Toner (SKU-1005) — relevance: -3.835
   • NOVA Hyaluronic Acid Serum (SKU-1006) — relevance: -7.306
⚡ Latency: 1297.4 ms

📨 Query 2: What sizes are available for the wrap dress?
-----------------------------------------------------------------
💬 Answer:
   The NOVA Wrap Dress is available in sizes XS, S, M, L, XL, and XXL. You can find these sizes listed in the product information for the NOVA Wrap Dress. The dres

In [13]:
# Generate 10 Q&A pairs using Groq for evaluation
print("⏳ Generating evaluation Q&A pairs using LLM...")

eval_generation_prompt = """
Generate 10 question-answer pairs for evaluating a RAG system for NOVA beauty brand.
Questions should be about: skincare ingredients, sizing, product compatibility,
return policy, shipping, and product recommendations.

Return ONLY valid JSON:
{
  "qa_pairs": [
    {
      "question": "Does the vitamin C serum work for oily skin?",
      "ground_truth": "Yes, the NOVA Vitamin C Serum is suitable for oily and combination skin types."
    }
  ]
}
"""

eval_response = client.chat.completions.create(
    model      = "llama-3.3-70b-versatile",
    messages   = [{"role": "user", "content": eval_generation_prompt}],
    max_tokens = 1500,
    temperature = 0.7,
    response_format={"type": "json_object"}
)

eval_data    = json.loads(eval_response.choices[0].message.content)
qa_pairs     = eval_data["qa_pairs"]

print(f"✅ Generated {len(qa_pairs)} evaluation Q&A pairs")
for i, qa in enumerate(qa_pairs, 1):
    print(f"   Q{i}: {qa['question'][:70]}...")

⏳ Generating evaluation Q&A pairs using LLM...
✅ Generated 10 evaluation Q&A pairs
   Q1: Does the vitamin C serum work for oily skin?...
   Q2: What is the size of the NOVA Hydrating Moisturizer?...
   Q3: Can I use the NOVA Retinol Night Cream with the NOVA Vitamin C Serum?...
   Q4: What is the return policy for NOVA beauty products?...
   Q5: How long does shipping take for NOVA beauty products?...
   Q6: What products would you recommend for acne-prone skin?...
   Q7: Is the NOVA Hydrating Moisturizer suitable for sensitive skin?...
   Q8: Can I exchange a product if it doesn't fit my skin type?...
   Q9: How do I track my NOVA beauty product order?...
   Q10: What is the difference between the NOVA Day Cream and the NOVA Night C...


In [14]:
print("⏳ Running RAG pipeline on evaluation questions...")

# Run retrieval for each eval question
eval_questions  = []
eval_answers    = []
eval_contexts   = []
eval_ground     = []

for qa in qa_pairs:
    result = retrieve_and_answer(qa["question"])
    eval_questions.append(qa["question"])
    eval_answers.append(result["answer"])
    eval_contexts.append([result["context"]])   # RAGAS expects list of strings
    eval_ground.append(qa["ground_truth"])
    print(f"   ✓ {qa['question'][:60]}...")

print(f"\n⏳ Running RAGAS evaluation metrics...")
print("   This may take 2-3 minutes...")

# Build RAGAS dataset
ragas_dataset = Dataset.from_dict({
    "question"   : eval_questions,
    "answer"     : eval_answers,
    "contexts"   : eval_contexts,
    "ground_truth": eval_ground
})

# Run evaluation
try:
    from langchain_groq import ChatGroq
    from langchain_community.embeddings import HuggingFaceEmbeddings

    ragas_llm = ChatGroq(
        groq_api_key = GROQ_API_KEY,
        model_name   = "llama-3.3-70b-versatile"
    )

    ragas_embeddings = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-small-en-v1.5"
    )

    results = evaluate(
        dataset    = ragas_dataset,
        metrics    = [faithfulness, answer_relevancy, context_precision],
        llm        = ragas_llm,
        embeddings = ragas_embeddings
    )

    scores = {
        "faithfulness"      : round(float(results["faithfulness"]), 4),
        "answer_relevancy"  : round(float(results["answer_relevancy"]), 4),
        "context_precision" : round(float(results["context_precision"]), 4),
    }

except Exception as e:
    print(f"⚠️  RAGAS auto-eval error: {e}")
    print("   Running manual scoring fallback...")

    # Manual scoring fallback using Groq
    scores_list = []
    for q, a, c in zip(eval_questions, eval_answers, eval_contexts):
        score_prompt = f"""Rate this RAG answer from 0.0 to 1.0.
Question: {q}
Context: {c[0][:300]}
Answer: {a}

Return JSON only: {{"faithfulness": 0.0-1.0, "relevancy": 0.0-1.0}}"""

        score_resp = client.chat.completions.create(
            model    = "llama-3.3-70b-versatile",
            messages = [{"role": "user", "content": score_prompt}],
            max_tokens = 50,
            response_format={"type": "json_object"}
        )
        s = json.loads(score_resp.choices[0].message.content)
        scores_list.append(s)

    scores = {
        "faithfulness"     : round(np.mean([s["faithfulness"] for s in scores_list]), 4),
        "answer_relevancy" : round(np.mean([s["relevancy"] for s in scores_list]), 4),
        "context_precision": round(0.0, 4)
    }

print(f"\n📊 RAGAS EVALUATION RESULTS")
print("=" * 40)
print(f"   Faithfulness      : {scores['faithfulness']}")
print(f"   Answer Relevancy  : {scores['answer_relevancy']}")
print(f"   Context Precision : {scores['context_precision']}")
print("=" * 40)

⏳ Running RAG pipeline on evaluation questions...
   ✓ Does the vitamin C serum work for oily skin?...
   ✓ What is the size of the NOVA Hydrating Moisturizer?...
   ✓ Can I use the NOVA Retinol Night Cream with the NOVA Vitamin...
   ✓ What is the return policy for NOVA beauty products?...
   ✓ How long does shipping take for NOVA beauty products?...
   ✓ What products would you recommend for acne-prone skin?...
   ✓ Is the NOVA Hydrating Moisturizer suitable for sensitive ski...
   ✓ Can I exchange a product if it doesn't fit my skin type?...
   ✓ How do I track my NOVA beauty product order?...
   ✓ What is the difference between the NOVA Day Cream and the NO...

⏳ Running RAGAS evaluation metrics...
   This may take 2-3 minutes...
⚠️  RAGAS auto-eval error: No module named 'langchain_groq'
   Running manual scoring fallback...

📊 RAGAS EVALUATION RESULTS
   Faithfulness      : 0.55
   Answer Relevancy  : 0.615
   Context Precision : 0.0


In [15]:
evaluation_report = {
    "metadata": {
        "task"          : "Task 3 — RAG Pipeline",
        "timestamp"     : datetime.datetime.now().isoformat(),
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "reranker_model" : "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "llm"            : "llama-3.3-70b-versatile",
        "total_documents": len(all_docs),
        "eval_questions" : len(qa_pairs)
    },
    "retrieval_config": {
        "strategy"    : "hybrid",
        "dense_top_k" : 10,
        "sparse_top_k": 10,
        "final_top_k" : 3,
        "dense_model" : "ChromaDB + BAAI/bge-small-en-v1.5",
        "sparse_model": "BM25Okapi"
    },
    "ragas_scores"  : scores,
    "test_results"  : [
        {
            "query"   : r["query"],
            "answer"  : r["answer"],
            "sources" : r["sources"],
            "latency_ms": r["latency_ms"]
        }
        for r in rag_test_results
    ]
}

with open("evaluation_report.json", "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("✅ evaluation_report.json saved")

✅ evaluation_report.json saved


In [16]:
rag_module_code = '''"""
rag_module.py — NOVA Product Knowledge RAG
Importable module used by Task 5 (Multi-Agent Platform)

Usage:
    from rag_module import retrieve_and_answer, build_rag_pipeline
"""

import os
import json
import time
import numpy as np
from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# ── Config ────────────────────────────────────────────────────
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHROMA_PATH     = "./chroma_db"
DB_PATH         = "./nova_mock_db.json"

# ── Global state (loaded once) ────────────────────────────────
_embedding_model = None
_reranker        = None
_collection      = None
_bm25_index      = None
_all_docs        = None
_client          = None


def _load_components(groq_api_key: str):
    """Load all RAG components into memory (called once)."""
    global _embedding_model, _reranker, _collection
    global _bm25_index, _all_docs, _client

    if _collection is not None:
        return  # Already loaded

    _client          = Groq(api_key=groq_api_key)
    _embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    _reranker        = CrossEncoder(RERANKER_MODEL)

    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    _collection   = chroma_client.get_collection("nova_products")

    with open(DB_PATH, "r") as f:
        db = json.load(f)

    _all_docs = []
    for p in db["products"]:
        _all_docs.append({"doc_id": p["product_id"],
                          "text": f"Product: {p[\'name\']}\\n{p[\'description\']}",
                          "metadata": {"product_id": p["product_id"],
                                       "name": p["name"],
                                       "category": p["category"]}})
    for f in db["faqs"]:
        _all_docs.append({"doc_id": f["id"],
                          "text": f"FAQ: {f[\'question\']}\\nAnswer: {f[\'answer\']}",
                          "metadata": {"product_id": f["id"],
                                       "name": f["question"],
                                       "category": f["category"]}})

    tokenized  = [d["text"].lower().split() for d in _all_docs]
    _bm25_index = BM25Okapi(tokenized)


def retrieve_and_answer(query: str, groq_api_key: str = None,
                        top_k: int = 3) -> dict:
    """
    Main RAG entry point for Task 5.
    Performs hybrid retrieval and generates a grounded answer.
    """
    if groq_api_key:
        _load_components(groq_api_key)

    start           = time.time()
    query_embedding = _embedding_model.encode(
        query, normalize_embeddings=True).tolist()

    dense_results = _collection.query(
        query_embeddings=[query_embedding], n_results=10,
        include=["documents", "metadatas", "distances"])

    dense_docs = [{"text": d, "metadata": m,
                   "score": 1 - dist, "source": "dense"}
                  for d, m, dist in zip(
                      dense_results["documents"][0],
                      dense_results["metadatas"][0],
                      dense_results["distances"][0])]

    bm25_scores  = _bm25_index.get_scores(query.lower().split())
    top_bm25_idx = np.argsort(bm25_scores)[::-1][:10]
    max_score    = max(bm25_scores) if max(bm25_scores) > 0 else 1

    sparse_docs = [{"text": _all_docs[i]["text"],
                    "metadata": _all_docs[i]["metadata"],
                    "score": bm25_scores[i] / max_score,
                    "source": "sparse"}
                   for i in top_bm25_idx if bm25_scores[i] > 0]

    seen, merged = set(), []
    for doc in dense_docs + sparse_docs:
        pid = doc["metadata"]["product_id"]
        if pid not in seen:
            seen.add(pid)
            merged.append(doc)

    if not merged:
        return {"query": query, "answer": "No relevant products found.",
                "sources": [], "context": "", "latency_ms": 0}

    pairs         = [[query, d["text"]] for d in merged]
    rerank_scores = _reranker.predict(pairs)

    for doc, score in zip(merged, rerank_scores):
        doc["rerank_score"] = float(score)

    top_docs = sorted(merged, key=lambda x: x["rerank_score"],
                      reverse=True)[:top_k]
    context  = "\\n\\n".join(
        f"[Source {i+1}]\\n{d[\'text\']}"
        for i, d in enumerate(top_docs))

    response = _client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system",
             "content": "Answer using ONLY the product info provided. Be concise."},
            {"role": "user",
             "content": f"Question: {query}\\n\\nProduct Info:\\n{context}"}
        ],
        max_tokens=300, temperature=0.2)

    return {
        "query"     : query,
        "answer"    : response.choices[0].message.content,
        "sources"   : [{"product_id": d["metadata"]["product_id"],
                        "name": d["metadata"]["name"],
                        "rerank_score": round(d["rerank_score"], 3)}
                       for d in top_docs],
        "context"   : context,
        "latency_ms": round((time.time() - start) * 1000, 2)
    }
'''

with open("rag_module.py", "w") as f:
    f.write(rag_module_code)

print("✅ rag_module.py saved")

✅ rag_module.py saved


In [17]:
from google.colab import files

files.download("evaluation_report.json")
files.download("rag_module.py")

print("✅ Files downloaded — push to GitHub:")
print("   → evaluation_report.json")
print("   → rag_module.py")
print("   → task3_rag_pipeline.ipynb  (File > Download > .ipynb)")
print("   → chroma_db/  (zip and upload to GitHub)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded — push to GitHub:
   → evaluation_report.json
   → rag_module.py
   → task3_rag_pipeline.ipynb  (File > Download > .ipynb)
   → chroma_db/  (zip and upload to GitHub)
